# Utility Gas Anomaly Detection

Flags unusual natural-gas consumption: yearly consumption patterns are clustered with dynamic
time warping (DTW), then anomalous years and months are flagged with an ensemble of clustering,
IQR, and standard-deviation methods.

Stack: pandas, scikit-learn, tslearn, MLflow.

Note: account and site identifiers are anonymized, and outputs are cleared because the
originals ran on internal data.


In [ ]:
!pip install openpyxl

In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from datetime import datetime, timedelta
from pyspark.sql import SparkSession

from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

In [ ]:
from datetime import datetime
import mlflow

# Set an experiment name
experiment_name = experiment_name = f"/Users/analyst/Utility_Anomaly_Detection"

# Check if the experiment already exists
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)

# Set the experiment to use
mlflow.set_experiment(experiment_name)

# Start an MLflow run and get the run ID
run = mlflow.start_run()
run_id = run.info.run_id

# Set the run name and tags
run_name = f"Gas_Run_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
mlflow.set_tag("mlflow.runName", run_name)
mlflow.set_tag("utility", "gas")

# Loading / Cleaning data

In [ ]:
# Load Gas Input (old one and new one)
data1 = pd.read_excel("/dbfs/FileStore/tables/utility_history/Gas_Input.xlsx")
data2 = pd.read_excel("/dbfs/mnt/adls-bidaadlsprd1/utility/gas.xlsx")

# Define the key features to check for duplicates
key_features = ['Cost Center', 'Service Acct#', 'Document #']

# Find rows in data2 that are duplicates based on key features in data1
duplicates_mask = data2[key_features].apply(tuple, axis=1).isin(data1[key_features].apply(tuple, axis=1))

# Remove duplicates from data2
data2_cleaned = data2[~duplicates_mask]

# Concatenate the cleaned data2 with data1
combined_data = pd.concat([data1, data2_cleaned])

# Reset index if necessary
combined_data.reset_index(drop=True, inplace=True)

In [ ]:
# Clean up data
gas = combined_data[['Cost Center', 'Service Acct#', 'Read From Date', 'Read To Date', 'Consumption M3', 'Vendor Name']].copy()
gas.rename(columns={'Cost Center':'site', 'Service Acct#':'acc', 'Read From Date': 'from', 'Read To Date': 'to', 'Consumption M3': 'consumption'}, inplace=True)

# Convert to datetime
gas[['from', 'to']] = gas[['from', 'to']].apply(pd.to_datetime)

# Fixing some data entry errors
# gas.loc[12427, 'to'] = pd.to_datetime('2016-11-25')
# gas.loc[46103, 'to'] = pd.to_datetime('2016-03-21')
# gas.loc[46105, 'to'] = pd.to_datetime('2016-03-21')
# gas.loc[51384, 'to'] = pd.to_datetime('2016-11-25')

# Calculate days of consumption period
gas['days'] = (gas['to'] - gas['from']).dt.days + 1 # add 1 to include the start day
gas['consumption_per_day'] = gas['consumption'] / gas['days']

gas_error = gas.loc[(gas['days'] < 0) | (gas['days'] > 184)]
gas = gas[(gas['days'] >= 0) & (gas['days'] <= 184)].copy()

display(combined_data.head(2))
gas.head(2)

In [ ]:
# Convert the timestamps to strings
read_from_date_min = gas['from'].min().strftime('%Y-%m-%d %H:%M:%S')
read_to_date_max = gas['to'].max().strftime('%Y-%m-%d %H:%M:%S')

# Logging date range metrics as strings
mlflow.log_param("read_from_date_min", read_from_date_min)
mlflow.log_param('read_to_date_max', read_to_date_max)

In [ ]:
# Expanding the dataframe on dates (consumption per day)

# drop the rows with any missing values
gas.dropna(inplace=True)

# Create a date range for each row and calculate the number of days
gas['date_range'] = gas.apply(lambda row: pd.date_range(row['from'], row['to']), axis=1)


# Explode the date_range to have a separate row for each date and assign daily consumption
gas = gas.explode('date_range')
gas['date'] = gas['date_range']
gas = gas.drop(['from', 'to', 'date_range', 'consumption', 'days'], axis=1)

gas.reset_index(drop=True, inplace=True)

gas.shape  # Display the first few rows of the expanded dataframe

In [ ]:
# calculate monthly consumption (sum of daily consumption)
gas_monthly_raw = gas.groupby(['site', 'acc', 'Vendor Name', pd.Grouper(key='date', freq='M')])['consumption_per_day'].sum().reset_index()

# Rename the columns for clarity
gas_monthly_raw.rename(columns={ 'consumption_per_day': 'monthly_consumption'}, inplace=True)

gas_monthly_raw.head(5)

In [ ]:
gas_monthly_raw.date.min(), gas_monthly_raw.date.max()

# Clustering -Dynamic Time Wraping (DTW)

In [ ]:
import datetime

df = gas_monthly_raw.copy()

# Creating a unique identifier for each site-account combination and preparing for standardization
df['id'] = df['site'].astype(int).astype(str) + '-' + df['acc'].astype(str)

# Date features
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# max year in the dataset
max_year = df['year'].max()

# max month of the max year in the dataset
max_month = df[df['year'] == max_year]['month'].max()

# Group by 'id' and 'year', and count the number of records for each group
monthly_counts = df.groupby(['id', 'year']).size().reset_index(name='num_months')

# Separate the current year (2024) data
current_year = df[df['year'] == max_year]
other_years = df[df['year'] != max_year]

# Filter groups to only include those with more than 8 data points for years other than the current year
sufficient_data = monthly_counts[(monthly_counts['num_months'] > 9) | (monthly_counts['year'] == max_year)]

# Merge the original DataFrame with the filtered groups to retain only rows that match
filtered_df = pd.merge(df, sufficient_data, on=['id', 'year'])

# Drop the 'num_months' column
filtered_df.drop('num_months', axis=1, inplace=True)

# Pivot the table to get consumption for each account and year, with months as columns
df_pivoted = filtered_df.pivot_table(index=['id', 'year'], columns='month', values='monthly_consumption', aggfunc='sum')

# Calculate the average consumption for each month across previous years for each account
monthly_averages = df_pivoted[df_pivoted.index.get_level_values('year') != max_year].groupby('id').mean()

# Identify the rows corresponding to the current year
current_year_pivoted = df_pivoted.loc[df_pivoted.index.get_level_values('year') == max_year]

for id_year in current_year_pivoted.index:
    id, year = id_year
    for month in range(max_month + 1, 13): 
        if pd.isna(current_year_pivoted.at[id_year, month]):
            # Check if the id exists in monthly_averages before attempting to access it
            if id in monthly_averages.index:
                current_year_pivoted.at[id_year, month] = monthly_averages.at[id, month]
            else:
                # If the id does not exist in monthly_averages, assign a fallback value (e.g., 0)
                current_year_pivoted.at[id_year, month] = 0

# Combine back the processed current year data with other years
final_df_pivoted = pd.concat([df_pivoted.loc[df_pivoted.index.get_level_values('year') != max_year], current_year_pivoted])

# Replace NaN values that appear due to missing months with 0 for other years
final_df_pivoted.fillna(0, inplace=True)

# Ensure the DataFrame is sorted by account and year if it's not already
final_df_pivoted.sort_index(inplace=True)


In [ ]:
from sklearn.preprocessing import StandardScaler


df_pivoted_standardized = final_df_pivoted.reset_index().copy()

scaler = StandardScaler()

# Scale all months in an account relative to each other
for id in df_pivoted_standardized['id'].unique():
    # Select all month columns for this account
    acc_data = df_pivoted_standardized.loc[df_pivoted_standardized['id'] == id, df_pivoted_standardized.columns[2:]]  # Adjust indexing as necessary
    
    # Flatten the account's monthly data to treat all months as a single distribution
    flattened_acc_data = acc_data.values.flatten().reshape(-1, 1)
    
    # Standardize this flattened data
    standardized_flattened_acc_data = scaler.fit_transform(flattened_acc_data).reshape(-1)
    
    # Assign the standardized values back to the DataFrame, maintaining the original shape
    df_pivoted_standardized.loc[df_pivoted_standardized['id'] == id, df_pivoted_standardized.columns[2:]] = standardized_flattened_acc_data.reshape(acc_data.shape)

# Note: This process treats all months within an account as part of a single distribution for standardization.
# It contrasts with scaling each month individually across years or scaling each month across all accounts.


In [ ]:
# Visualizing the yearly consumption patterns
plt.figure(figsize=(12, 8))

# Define the months range for the x-axis
months = range(1, 13)

# Loop through each account to plot its monthly consumption pattern
# Set a single color for all lines and use the alpha parameter to control transparency
for id in df_pivoted_standardized['id'].unique():
    for year in df_pivoted_standardized[(df_pivoted_standardized['id'] == id)]['year'].unique():
        acc_data = df_pivoted_standardized[(df_pivoted_standardized['id'] == id) & (df_pivoted_standardized['year'] == year)]
        plt.plot(months, acc_data.iloc[0, 2:].values, marker='o', linestyle='-', color='blue', alpha=0.01)  # Semi-transparent lines
    acc_data = df_pivoted_standardized[df_pivoted_standardized['id'] == id]

plt.xlabel('Month')
plt.ylabel('Standardized Monthly Consumption')
plt.title('Standardized Annual Consumption Patterns Across Accounts')
plt.xticks(months)  # Ensure all months are shown
plt.grid(True)

# Omit the legend to keep the focus on the overall pattern density
# plt.legend()  # Commented out due to potential clutter with many lines

plt.show()

In [ ]:
# Reshape for tslearn which expects a 3D array: (n_series, series_length, n_features)
time_series = df_pivoted_standardized.loc[:,1:12].values.reshape((final_df_pivoted.shape[0], final_df_pivoted.shape[1], 1))

In [ ]:
# Clustering with DTW

# Define the number of clusters
n_clusters = 3 # Adjust as needed

# Initialize the TimeSeriesKMeans model with DTW metric
model = TimeSeriesKMeans(n_clusters=n_clusters, 
                         n_init=10,
                         metric="softdtw", 
                         verbose=True, 
                         max_iter_barycenter=40,
                         metric_params={"gamma": 1.6}, # Soft-DTW parameter (A higher gamma value makes the clustering less sensitive to small variations in the time series)
                         n_jobs=-1,  # Use all available CPU cores
                         random_state=1)

# Fit the model and predict clusters
clusters = model.fit_predict(time_series)

In [ ]:
# compute silhouette_score to evaluate the clustering performance
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
silhouette_avg  = silhouette_score(time_series.reshape((time_series.shape[0], time_series.shape[1])), clusters, metric='euclidean')
davies_bouldin = davies_bouldin_score(time_series.reshape((time_series.shape[0], time_series.shape[1])), clusters)
calinski_harabasz = calinski_harabasz_score(time_series.reshape((time_series.shape[0], time_series.shape[1])), clusters)

# log the model performance
mlflow.log_metric('silhouette_score', silhouette_avg)
mlflow.log_metric('davies_bouldin_score', davies_bouldin)
mlflow.log_metric('calinski_harabasz_score', calinski_harabasz)

print(f"Silhouette Score: {silhouette_avg:.2f}")
print(f"Davies-Bouldin Score: {davies_bouldin:.2f}")
print(f"Calinski-Harabasz Score: {calinski_harabasz:.2f}")

In [ ]:
# Revert the data points from June 2024 to December 2024 back to NaN after anomaly detection
for month in range(max_month + 1, 13):
    final_df_pivoted.loc[(slice(None), max_year), month] = float('nan')

In [ ]:
# Visualization of the cluster centroids
import matplotlib.pyplot as plt

# Add the cluster labels back to the DataFrame
final_df_pivoted['cluster'] = clusters


for i, centroid in enumerate(model.cluster_centers_):
    plt.plot(centroid.ravel(), label=f'Cluster {i}')
plt.title('Cluster Centroids')
plt.xlabel('Month')
plt.ylabel('Standardized Consumption')
plt.legend()
plt.show()

In [ ]:
final_df_pivoted.cluster.value_counts()

In [ ]:
# Visualize Each Cluster Separately
import matplotlib.pyplot as plt
import numpy as np

# Add the cluster labels back to the standardized DataFrame
df_pivoted_standardized['cluster'] = clusters

# Define the months range for the x-axis
months = np.arange(1, 13)  # Assuming there are 12 months of data

# Sort clusters to plot in order
clust = np.sort(df_pivoted_standardized['cluster'].unique())

# Define colors for each cluster for visual distinction
colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black']  # Ensure there are enough colors for all clusters

# Calculate the number of unique clusters for subplot dimensions
n_clusters = len(clust)

# Create a figure for each cluster
for i, cluster in enumerate(clust):
    plt.figure(figsize=(12, 4))  # Adjust figure size as needed
    
    cluster_data = df_pivoted_standardized[df_pivoted_standardized['cluster'] == cluster]
    color = colors[cluster % len(colors)]  # Choose color based on cluster index
    
    for _, row in cluster_data.iterrows():
        # the monthly data starts at the 4th column and ends at the second to last column
        monthly_values = row.iloc[2:-1].values 
        plt.plot(months, monthly_values, marker='o', linestyle='-', color=color, alpha=0.1)
    
    plt.title(f'Cluster {cluster} Standardized Annual Consumption Patterns')
    plt.xlabel('Month')
    plt.ylabel('Standardized Monthly Consumption')
    plt.xticks(months)  # Ensure all months are shown
    plt.grid(True)
    
    # Show plot for the current cluster
    plt.show()


# Anomaly Detection

### Flag Anomalous Years for Each Account (in case of ties take the first mode)

In [ ]:
# Flagging the years with abnormal consumption patterns
df_cluster = final_df_pivoted.reset_index()

# Calculate the most frequent cluster for each account
most_frequent_cluster = df_cluster.groupby('id')['cluster'].agg(lambda x: pd.Series.mode(x)[0]).reset_index()
most_frequent_cluster.rename(columns={'cluster': 'most_frequent_cluster'}, inplace=True)

# Merge this back with the original DataFrame to compare each year's cluster to the most frequent cluster
df_cluster = df_cluster.merge(most_frequent_cluster, on='id')

# Flag years where the account's cluster is different from its most frequent cluster
df_cluster['is_anomaly'] = df_cluster.apply(lambda x: x['cluster'] != x['most_frequent_cluster'], axis=1)

df_cluster.is_anomaly.value_counts(normalize=True)

### Flag Anomalous Years for Each Account (weigted approach only in case of ties)

In [ ]:
#  Identify Accounts with Ties
# Aggregate to get the mode, ensuring the result is always a list, even if there's only one mode
mode_clusters = df_cluster.groupby('id')['cluster'].agg(lambda x: list(x.mode()))

# Now, since mode_clusters always contains lists, we can safely apply len()
ties = mode_clusters[mode_clusters.apply(len) > 1]  # Identifying accounts with ties

# For accounts with a clear mode (no ties), we still have lists but with a single element
no_ties = mode_clusters[mode_clusters.apply(len) == 1].apply(lambda x: x[0])  # Simplifying to get the value out of the list

# To handle ties by selecting the most recent year's cluster for those accounts,
# we need to identify these accounts in the original DataFrame
ties_index = ties.index  # Accounts with ties based on the index of the 'ties' Series


In [ ]:
# A simple weight could be the year itself, or a more complex scheme like exponential weighting based on recency
df_cluster['weight'] = df_cluster['year'] - df_cluster['year'].min() + 1

# For each account with ties, calculate the weighted cluster preference
weighted_preference = df_cluster[df_cluster['id'].isin(ties.index)].groupby(['id', 'cluster'])['weight'].sum().reset_index()

# Determine the cluster with the maximum weight for each account
weighted_preference['rank'] = weighted_preference.groupby('id')['weight'].rank(method='first', ascending=False)
preferred_cluster = weighted_preference[weighted_preference['rank'] == 1].drop('rank', axis=1)
preferred_cluster.rename(columns={'cluster': 'typical_cluster'}, inplace=True)

In [ ]:
df_cluster = pd.merge(df_cluster, preferred_cluster[['id', 'typical_cluster']], on=['id'], how='left')
# Fill NaN values in 'typical_cluster' with the values from 'most_frequent_cluster'
df_cluster['typical_cluster'].fillna(df_cluster['most_frequent_cluster'], inplace=True)
df_cluster['typical_cluster'] = df_cluster['typical_cluster'].astype(int)

# Flag years where the account's cluster is different from its typical cluster
df_cluster['is_anomaly'] = df_cluster.apply(lambda x: x['cluster'] != x['typical_cluster'], axis=1)

In [ ]:
df_cluster.is_anomaly.value_counts(normalize=True)

## Statistical Approach to Flag Abnormal Year

### IQR

In [ ]:
# Flagging the outlier months for each account using IQR method
df_iqr = final_df_pivoted.reset_index().copy()

# Ensure 'average_normal' is calculated correctly
average_normal = df_iqr.groupby('id').agg({i: 'mean' for i in range(1, 13)})

# Calculate Q1, Q3 for each month for each account during normal years
# Adjust the column selection to match your DataFrame's structure
def quantile_agg(x):
    return x.quantile(0.25), x.quantile(0.75)

# Apply aggregation to get Q1 and Q3 for each month
monthly_stats = df_iqr.groupby('id')[[i for i in range(1, 13)]].agg([lambda x: quantile_agg(x)[0], lambda x: quantile_agg(x)[1]])

# Rename the aggregated columns to Q1 and Q3 for clarity
monthly_stats.columns = [f'{month}_{stat}' for month in range(1, 13) for stat in ['Q1', 'Q3']]

# Calculate IQR for each month
for i in range(1, 13):
    monthly_stats[f'month_{i}_IQR'] = monthly_stats[f'{i}_Q3'] - monthly_stats[f'{i}_Q1']

# Function to identify outliers based on Q1, Q3, and IQR of normal years
def identify_outliers_with_threshold(row, quartiles, averages, threshold_percentage=10, min_abs_threshold=10):
    acc = row['id']
    outliers = []
    for month in range(1, 13):
        q1 = quartiles.at[acc, f'{month}_Q1']
        q3 = quartiles.at[acc, f'{month}_Q3']
        iqr = quartiles.at[acc, f'month_{month}_IQR']
        month_value = row[month]
        
        # Determine the minimum threshold based on the average of normal points
        avg_normal = averages.loc[acc, month]
        # Check if avg_normal is zero or close to zero and set min_threshold accordingly
        # If avg_normal is less than a very small number (e.g., 0.01), use min_abs_threshold instead
        if avg_normal < 0.01:
            min_threshold = min_abs_threshold
        else:
            min_threshold = avg_normal * (threshold_percentage / 100.0)
        
        # Identify outliers based on the threshold
        if month_value < (q1 - max(min_threshold, 1.5 * iqr)) or month_value > (q3 + max(min_threshold, 1.5 * iqr)):
            outliers.append(month)
    return outliers

# Assuming 'threshold_percentage' is the percentage of the average normal consumption
# that defines the minimum significant deviation
df_iqr['outlier_months'] = df_iqr.apply(lambda row: identify_outliers_with_threshold(row, monthly_stats, average_normal, threshold_percentage=50,  min_abs_threshold=100), axis=1)
df_cluster['is_anomaly_iqr'] = (df_iqr.outlier_months.apply(len)>0)

In [ ]:
df_cluster.is_anomaly_iqr.value_counts(normalize=True)

### Standard Deviation

In [ ]:
# Flagging the outlier months for each account using Standard Deviation method
df_sd = final_df_pivoted.reset_index().copy()

# Step 1: Calculate Mean and Standard Deviation for Normal Years
# Calculate mean and std for each month for each account during normal years
monthly_stats = df_sd.groupby('id').agg(['mean', 'std']).loc[:, 1:12]  # slicing for month columns only

# Step 2: Identify Outlier Months in Anomalous Years
# Function to identify outliers based on mean and std of normal years
def identify_outliers(row, stats):
    acc = row['id']  # Account identifier
    outliers = []  # To store outlier months
    for month in range(1, 13):
        # Skip if it's not an anomalous year
        month_mean = stats.loc[acc, (month, 'mean')]
        month_std = stats.loc[acc, (month, 'std')]
        # Calculate Z-score for the month
        if month_std > 0:  # Avoid division by zero
            z_score = (row[month] - month_mean) / month_std
            if abs(z_score) > 3 or abs(row[month]-month_mean)/month_mean>=1:  # Threshold for outliers
                outliers.append(month)
    return outliers

# Apply the function to each row in the DataFrame of anomalous years
df_sd['outlier_months'] = df_sd.apply(lambda row: identify_outliers(row, monthly_stats), axis=1)
df_cluster['is_anomaly_sd'] = (df_sd.outlier_months.apply(len)>0)

In [ ]:
df_cluster['is_anomaly_sd'].value_counts(normalize=True)

# Ensembling

In [ ]:
df_cluster['final_anomaly'] = (df_cluster.is_anomaly.astype(int) + df_cluster.is_anomaly_iqr.astype(int) + df_cluster.is_anomaly_sd.astype(int)) >= 2

In [ ]:
# loging anomaly percentage
mlflow.log_metric('anomaly_percentage', df_cluster.final_anomaly.mean())

# Flag Outlier Months in Anomalous Years

## Standard Deviation

In [ ]:
# Step 1: Calculate Mean and Standard Deviation for Normal Years

# Create a DataFrame for normal years only
normal_years_df = df_cluster[~df_cluster['final_anomaly']][['id', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]]

# Calculate mean and std for each month for each account during normal years
monthly_stats = normal_years_df.groupby('id').agg(['mean', 'std'])  # slicing for month columns only

# Ensure the monthly_stats DataFrame has a multi-level column index [month, stat_type]
#monthly_stats.columns = pd.MultiIndex.from_product([range(1, 13), ['mean', 'std']], names=['month', 'stat_type'])

In [ ]:
# Step 2: Identify Outlier Months in Anomalous Years

# Identify accounts for which all the years are anomalous (i.e., no normal years)
all_anomaly_ids = df_cluster.groupby('id')['final_anomaly'].value_counts().unstack().reset_index().rename(columns={False: 'normal', True: 'anomaly'}).query('normal.isna()')['id']

# Function to identify outliers based on mean and std of normal years
def identify_outliers(row, stats):
    id = row['id']  # Account identifier
    outliers = []  # To store outlier months
    # Skip if id is in all_anomaly_ids
    if id in all_anomaly_ids.values:
        return []
    else:
        for month in range(1, 13):
            # Skip if it's not an anomalous year
            if not row['final_anomaly']: continue
            month_mean = stats.loc[id, (month, 'mean')]
            month_std = stats.loc[id, (month, 'std')]
            # Calculate Z-score for the month
            if month_std > 0:  # Avoid division by zero
                z_score = (row[month] - month_mean) / month_std
                if abs(z_score) > 3:  # Threshold for outliers
                    outliers.append(month)
        return outliers

# Apply the function to each row in the DataFrame of anomalous years
df_cluster['outlier_months'] = df_cluster.apply(lambda row: identify_outliers(row, monthly_stats), axis=1)

## IQR

In [ ]:
# Create a DataFrame for normal years only
normal_years_df = df_cluster[~df_cluster['final_anomaly']]

# Identify accounts for which all the years are anomalous (i.e., no normal years)
all_anomaly_ids = df_cluster.groupby('id')['final_anomaly'].value_counts().unstack().reset_index().rename(columns={False: 'normal', True: 'anomaly'}).query('normal.isna()')['id']

# Ensure 'average_normal' is calculated correctly
average_normal = normal_years_df.groupby('id').agg({i: 'mean' for i in range(1, 13)})

# Calculate Q1, Q3 for each month for each account during normal years
# Adjust the column selection to match your DataFrame's structure
def quantile_agg(x):
    return x.quantile(0.25), x.quantile(0.75)

# Apply aggregation to get Q1 and Q3 for each month
monthly_stats = normal_years_df.groupby('id')[[i for i in range(1, 13)]].agg([lambda x: quantile_agg(x)[0], lambda x: quantile_agg(x)[1]])

# Rename the aggregated columns to Q1 and Q3 for clarity
monthly_stats.columns = [f'{month}_{stat}' for month in range(1, 13) for stat in ['Q1', 'Q3']]

# Calculate IQR for each month
for i in range(1, 13):
    monthly_stats[f'month_{i}_IQR'] = monthly_stats[f'{i}_Q3'] - monthly_stats[f'{i}_Q1']

# Function to identify outliers based on Q1, Q3, and IQR of normal years
def identify_outliers_with_threshold(row, quartiles, averages, threshold_percentage=10, min_abs_threshold=10):
    id = row['id']
    outliers = []
    # Skip if id is in all_anomaly_ids
    if id in all_anomaly_ids.values:
        return []
    else:
        for month in range(1, 13):
            if not row['final_anomaly']:
                continue
            q1 = quartiles.at[id, f'{month}_Q1']
            q3 = quartiles.at[id, f'{month}_Q3']
            iqr = quartiles.at[id, f'month_{month}_IQR']
            month_value = row[month]
            
            # Determine the minimum threshold based on the average of normal points
            avg_normal = averages.loc[id, month]
            # Check if avg_normal is zero or close to zero and set min_threshold accordingly
            # If avg_normal is less than a very small number (e.g., 0.01), use min_abs_threshold instead
            if avg_normal < 0.01:
                min_threshold = min_abs_threshold
            else:
                min_threshold = avg_normal * (threshold_percentage / 100.0)
            
            # Identify outliers based on the threshold
            if month_value < (q1 - max(min_threshold, 1.5 * iqr)) or month_value > (q3 + max(min_threshold, 1.5 * iqr)):
                outliers.append(month)
    return outliers

# Assuming 'threshold_percentage' is the percentage of the average normal consumption
# that defines the minimum significant deviation
df_cluster['outlier_months'] = df_cluster.apply(lambda row: identify_outliers_with_threshold(row, monthly_stats, average_normal, threshold_percentage=50,  min_abs_threshold=100), axis=1)

# Final Visualization

In [ ]:
# Specify the account and site to visualize
id = '40607-G120101740710'

# Filter the DataFrame for the specified account and site
df_plot = df_cluster[df_cluster['id'] == id]

# Plot the yearly consumption patterns with cluster labels, anomalies, and highlight outlier months
plt.figure(figsize=(12, 4))
for year in df_plot['year'].unique():
    year_data = df_plot[df_plot['year'] == year]
    color = 'red' if year_data['final_anomaly'].iloc[0] else 'blue'
    plt.plot(range(1, 13), year_data.iloc[0, 2:14], label=f'Year {year}', color=color)
    
    # If the year is an anomaly, highlight the outlier months
    if year_data['final_anomaly'].iloc[0]:
        outlier_months = year_data['outlier_months'].iloc[0]
        # Check if outlier_months is not NaN and not empty
        if outlier_months and not pd.isna(outlier_months).any():
            # Highlight each outlier month
            for month in outlier_months:
                plt.scatter(month, year_data.iloc[0, month + 1], color='yellow', s=100, edgecolors='black', zorder=5)

plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])  # Ensure all 12 months are displayed on the x-axis
plt.xlabel('Month')
plt.ylabel('Standardized Monthly Consumption')
plt.title(f'Standardized Annual Consumption Patterns for {id}')
plt.legend()
plt.grid(True)
plt.show()

cols = ['year', 1,2,3,4,5,6,7,8,9,10,11,12,'is_anomaly','is_anomaly_iqr', 'is_anomaly_sd', 'final_anomaly']
df_cluster[df_cluster.id == id][cols].round(1)

# Final Output

In [ ]:
# save the df_cluster to a csv file
final_output = df_cluster[['id', 'year', 1,2,3,4,5,6,7,8,9,10,11,12,'final_anomaly', 'outlier_months']].copy()

for month in range(1, 13):
    final_output[f'outlier_{month}'] = 0  # Initialize with 0

# Function to update the outlier flag based on the outlier_months list
def update_outlier_flags(row):
    for month in row['outlier_months']:
        if month:  # Check if there's a valid month entry
            row[f'outlier_{month}'] = 1
    return row

# Apply the function across the DataFrame
final_output = final_output.apply(update_outlier_flags, axis=1)

# Splitting the 'id' column into 'site_number' and 'account_number'
final_output[['site', 'acc']] = final_output['id'].str.split('-', n=1, expand=True)

# Extract vendor information from water_monthly_raw
vendor_info = gas_monthly_raw[['site', 'acc', 'Vendor Name']].drop_duplicates()

# Convert 'site' and 'acc' columns to string in both DataFrames to ensure matching data types
final_output['site'] = final_output['site'].astype(str)
final_output['acc'] = final_output['acc'].astype(str)
vendor_info['site'] = vendor_info['site'].astype(str)
vendor_info['acc'] = vendor_info['acc'].astype(str)

# Merge vendor information with final output
final_output_with_vendor = pd.merge(final_output, vendor_info, on=['site', 'acc'], how='left')

# change the column Vendor Name to vendor_name
final_output_with_vendor = final_output_with_vendor.rename(columns={'Vendor Name': 'vendor_name'})

# create a new bolean column that shows accounts with anomalous years as anomalous_accounts
final_output_with_vendor['anomalous_accounts'] = final_output_with_vendor.groupby('id')['final_anomaly'].transform('sum') > 0
final_output_with_vendor['anomalous_site'] = final_output_with_vendor.groupby('site')['final_anomaly'].transform('sum') > 0

final_output_with_vendor.head(3)

 # Save Data to Azure Data Lake

In [ ]:
# Create the schema if it does not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS utilities")

# Convert the Pandas DataFrame to a PySpark DataFrame
final_output_spark = spark.createDataFrame(final_output_with_vendor)

# Write the PySpark DataFrame as a Delta table
table_name = "utilities.gas_anomaly_detection"
final_output_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

In [ ]:
# Save Gas error Data to an Azure Data Lake

# chnage column name
gas_error = gas_error.rename(columns={'Vendor Name': 'vendor_name'})

# Convert the Pandas DataFrame to a PySpark DataFrame
gas_error_spark = spark.createDataFrame(gas_error)

# Write the PySpark DataFrame as a Delta table
table_name = "utilities.gas_data_entry_error"
gas_error_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

In [ ]:
gas_error.head()

In [ ]:
mlflow.end_run()